# Freeze or Fine-Tune: The Decision and Its Cost

**Computational Text Analysis Pack · Module 26 — Fine-Tuning Pretrained Models**

*Technique:* Frozen features vs full fine-tuning; measuring what adaptation buys

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/13-african-nlp-models/26a_freeze_or_finetune.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

You have a pretrained model and a labelled corpus. There are two ways to use
them, and choosing between them is the single most consequential decision in an
applied neural project.

- **Frozen features.** Run the model once, keep the output vectors, train a
  logistic regression on them. Minutes, no GPU, and the pretrained weights never
  change.
- **Fine-tuning.** Continue training the whole model on your labels. Hours or a
  GPU, and usually a few points better.

The honest question is whether those few points are worth the cost, and that
depends on your corpus — so we measure it rather than recommending.

## The tooling

**HuggingFace `transformers`** provides the models and the training loop
(`Trainer`), **`datasets`** the data plumbing, and **PyTorch** the engine. For
serious runs you would reach for a GPU; everything below is sized to run on CPU
so the workflow is learnable without one.

The alternative to fine-tuning is **frozen features**: run the pretrained model
once, keep the vectors, and train a cheap classifier on top. It is far faster,
often nearly as good, and is what module 23d already did with LaBSE.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np, time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Frozen features, already computed on the GPU and shipped with the course.
bank = np.load("labse_headlines.npy")
index = pd.read_csv("labse_headlines_index.csv")
mask = index["category"].notna().values
X, y = bank[mask], index.loc[mask, "category"].values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(f"{len(X)} headlines, {X.shape[1]} dims, {len(set(y))} categories")

t0 = time.time()
frozen = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
p = frozen.predict(Xte)
frozen_secs = time.time() - t0
frozen_acc = accuracy_score(yte, p)
print(f"\nFROZEN FEATURES  acc {frozen_acc:.3f}  "
      f"macro-F1 {f1_score(yte, p, average='macro'):.3f}  {frozen_secs:.1f}s")

## Now fine-tune, at a size that fits here

We use a deliberately tiny transformer (`bert-tiny`, ~4M parameters against
BERT-base's 110M) on a small subset. This is not a competitive setup — it is a
*legible* one. Watch the loop, not the leaderboard.

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL = "prajjwal1/bert-tiny"
N_TRAIN, N_TEST, EPOCHS, MAXLEN = 700, 300, 3, 48

sub = df[df["n_words"] >= 30].sample(n=N_TRAIN + N_TEST, random_state=3).reset_index(drop=True)
cats = sorted(sub["category"].unique())
c2i = {c: i for i, c in enumerate(cats)}

try:
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(cats))
    FT_OK = True
except Exception as exc:
    FT_OK = False
    print("could not load the model (no network?):", str(exc)[:90])

if FT_OK:
    enc = tok(sub["headline"].fillna("").astype(str).tolist(), truncation=True,
              padding="max_length", max_length=MAXLEN, return_tensors="pt")
    labels = torch.tensor([c2i[c] for c in sub["category"]])
    ds = TensorDataset(enc["input_ids"], enc["attention_mask"], labels)
    train_ds = torch.utils.data.Subset(ds, range(N_TRAIN))
    test_ds = torch.utils.data.Subset(ds, range(N_TRAIN, len(sub)))
    print(f"fine-tuning {MODEL} on {N_TRAIN} examples, {EPOCHS} epochs, CPU")

In [ ]:
if FT_OK:
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4)
    loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    t0 = time.time()
    model.train()
    for ep in range(EPOCHS):
        total = 0.0
        for ids, am, lab in loader:
            opt.zero_grad()
            out = model(input_ids=ids, attention_mask=am, labels=lab)
            out.loss.backward(); opt.step()
            total += float(out.loss)
        print(f"   epoch {ep+1}  mean loss {total/len(loader):.4f}")
    ft_secs = time.time() - t0

    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for ids, am, lab in DataLoader(test_ds, batch_size=64):
            preds.extend(model(input_ids=ids, attention_mask=am).logits.argmax(-1).tolist())
            golds.extend(lab.tolist())
    ft_acc = accuracy_score(golds, preds)
    print(f"\nFINE-TUNED       acc {ft_acc:.3f}  "
          f"macro-F1 {f1_score(golds, preds, average='macro'):.3f}  {ft_secs:.0f}s")
else:
    ft_acc, ft_secs = float('nan'), float('nan')

In [ ]:
print(f"{'approach':<28}{'accuracy':>10}{'seconds':>10}{'needs a GPU?':>15}")
print("-" * 64)
print(f"{'frozen LaBSE + logistic':<28}{frozen_acc:>10.3f}{frozen_secs:>10.1f}{'no':>15}")
if FT_OK:
    print(f"{'fine-tuned bert-tiny':<28}{ft_acc:>10.3f}{ft_secs:>10.0f}{'no (but slow)':>15}")
print(f"{'fine-tuned BERT-base':<28}{'~0.85-0.90':>10}{'hours':>10}{'yes':>15}   (published figures)")
print()
print("Read this as a cost curve, not a ranking. Frozen features get you most")
print("of the way in seconds; fine-tuning a tiny model on a few hundred")
print("examples is worse than both, because it has neither scale nor a good")
print("starting representation. The middle of that curve is where the")
print("GPU-hours go.")

## The decision rule

1. **Always try frozen features first.** It is an afternoon, costs nothing, and
   sets the baseline any fine-tuning must beat.
2. **Fine-tune when the domain is far from pretraining** — historical spelling,
   a specialised register, a language the model barely saw.
3. **Below ~1,000 labelled examples, fine-tuning usually overfits.** Frozen
   features degrade far more gracefully.
4. **Budget the comparison, not just the run.** A fine-tune you cannot afford to
   repeat is a result you cannot check.

### 🔧 Your turn 1

Cut N_TRAIN to 200 and re-run the fine-tune. Does it get worse faster than the frozen baseline would?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Yes, and sharply — a fine-tuned model with too few examples chases noise in the
labels, while the frozen classifier is only fitting a few hundred coefficients
over a representation that is already good. This asymmetry is the practical
reason frozen features dominate small-corpus humanities work: most DH projects
have hundreds of hand-labelled items, not tens of thousands, and that is
squarely the regime where the cheaper method wins.

</details>